# 18. Structural detector: fine-tune, ablation, and evaluation on the deployment distribution

Trains a payload-decoupled structural detector (DeBERTa-v3-base) on the notebook-17 dataset and evaluates it where the surviving finding lives: hijack and harmful injections embedded in BIPIA documents, against BIPIA benign hosts. GPU for training and scoring; every score is cached under `data/structural/scores/`, so re-runs are CPU only.

**Three training configurations**, so the ablation is not confounded with data size:

| config | positives | negatives |
|---|---|---|
| full | embedded | standalone, host_only, declarative_insert, content_imperative_insert |
| ablation_no_twins | embedded | host_only, declarative_insert only (no instruction-bearing twins) |
| full_sizematched | embedded | all four forms, subsampled to the ablation's negative count |

**Evaluation.** AUROC and partial AUROC (FPR up to 0.05) for hijack and harmful documents against hosts, the interval protocol with the 778-host document pool as calibration (stable, from notebook 15) and with deepset benigns (the preprint's protocol), NotInject over-defense, rank severity R of the misses, and the held-out structural test split per form. Comparators: the three released detectors from cache and a TF-IDF reference trained on the same data. The direct and jailbreak sets are reported too: a detector of instructions inside untrusted content is not built for standalone prompts, and the table should say so rather than hide it.

**Outputs.** `reports/structural_detector_eval.csv`, `reports/structural_detector_interval.csv`, `reports/structural_detector_testsplit.csv`, `figures/structural_detector_roc.png`.

In [ ]:
# === SESSION BOOTSTRAP (run first) ===
from google.colab import drive
drive.mount('/content/drive')
import os, sys
DRIVE_ROOT = '/content/drive/MyDrive/PICALIB_Research'
REPO_DIR   = os.path.join(DRIVE_ROOT, 'picalib-research')
!git config --global user.name  "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git config --global credential.helper "store --file={DRIVE_ROOT}/.git-credentials"
%cd {REPO_DIR}
sys.path.insert(0, 'src')
!git pull
!pip install -q transformers datasets accelerate sentencepiece
os.makedirs('reports', exist_ok=True); os.makedirs('figures', exist_ok=True); os.makedirs('data/structural/scores', exist_ok=True)
import torch; print('Session ready. GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE (training will be very slow)')

## Stage A. Training module

In [ ]:
%%writefile src/structural_train.py
"""Fine-tune and score a payload-decoupled structural detector on the notebook-17 dataset."""
from __future__ import annotations
import json, os, inspect, random
import numpy as np

MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LEN = 512


def load_jsonl(path):
    return [json.loads(l) for l in open(path, encoding="utf-8")]


def select_config(records, config, rng):
    """Return the training records for a config.
    full:              all five forms
    ablation_no_twins: positives + negatives from host_only and declarative_insert only
    full_sizematched:  all five forms, negatives subsampled to the ablation's negative count, form-proportional
    """
    pos = [r for r in records if r["label"] == 1]
    neg = [r for r in records if r["label"] == 0]
    if config == "full":
        return pos + neg
    if config == "ablation_no_twins":
        return pos + [r for r in neg if r["form"] in ("host_only", "declarative_insert")]
    if config == "full_sizematched":
        target = sum(r["form"] in ("host_only", "declarative_insert") for r in neg)
        by_form = {}
        for r in neg: by_form.setdefault(r["form"], []).append(r)
        total = len(neg); out = []
        for form, rs in by_form.items():
            k = int(round(target * len(rs) / total)); rng.shuffle(rs); out += rs[:k]
        return pos + out
    raise ValueError(config)


def class_weights(records):
    n1 = sum(r["label"] for r in records); n0 = len(records) - n1
    return [len(records) / (2 * n0), len(records) / (2 * n1)]


def train_model(train_records, val_records, out_dir, seed=0, epochs=2, lr=2e-5, batch=8, accum=2, max_len=MAX_LEN):
    """Fine-tune with a class-weighted loss; select the epoch with the best val AUROC. Returns out_dir."""
    import torch
    from datasets import Dataset
    from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, set_seed)
    from sklearn.metrics import roc_auc_score
    set_seed(seed)
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    def enc(b): return tok(b["text"], truncation=True, max_length=max_len)
    tr = Dataset.from_list([{"text": r["text"], "label": int(r["label"])} for r in train_records]).map(enc, batched=True, remove_columns=["text"])
    va = Dataset.from_list([{"text": r["text"], "label": int(r["label"])} for r in val_records]).map(enc, batched=True, remove_columns=["text"])
    w = torch.tensor(class_weights(train_records), dtype=torch.float)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

    class WeightedTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.pop("labels")
            outputs = model(**inputs)
            loss = torch.nn.functional.cross_entropy(outputs.logits, labels, weight=w.to(outputs.logits.device))
            return (loss, outputs) if return_outputs else loss

    def metrics(ep):
        logits, labels = ep.predictions, ep.label_ids
        if isinstance(logits, (tuple, list)): logits = logits[0]
        p = torch.softmax(torch.tensor(np.asarray(logits)), -1)[:, 1].numpy()
        return {"auroc": float(roc_auc_score(labels, p))}

    params = inspect.signature(TrainingArguments.__init__).parameters
    strat_key = "eval_strategy" if "eval_strategy" in params else "evaluation_strategy"
    import math
    total_steps = math.ceil(len(tr) / (batch * accum)) * epochs
    warm = {"warmup_ratio": 0.06} if "warmup_ratio" in params else {"warmup_steps": int(0.06 * total_steps)}   # transformers 5.x dropped warmup_ratio
    args = TrainingArguments(output_dir=out_dir, per_device_train_batch_size=batch, per_device_eval_batch_size=32,
                             gradient_accumulation_steps=accum, num_train_epochs=epochs, learning_rate=lr, weight_decay=0.01,
                             fp16=torch.cuda.is_available(), save_strategy="epoch", load_best_model_at_end=True,
                             metric_for_best_model="auroc", greater_is_better=True, save_total_limit=1, logging_steps=50,
                             report_to="none", seed=seed, **warm, **{strat_key: "epoch"})
    tok_key = "processing_class" if "processing_class" in inspect.signature(Trainer.__init__).parameters else "tokenizer"
    trainer = WeightedTrainer(model=model, args=args, train_dataset=tr, eval_dataset=va, compute_metrics=metrics, **{tok_key: tok})
    trainer.train()
    trainer.save_model(os.path.join(out_dir, "best")); tok.save_pretrained(os.path.join(out_dir, "best"))
    json.dump({"best_metric": trainer.state.best_metric, "log": trainer.state.log_history}, open(os.path.join(out_dir, "train_log.json"), "w"))
    return os.path.join(out_dir, "best")


def score_texts(model_dir, texts, batch=32, max_len=MAX_LEN):
    """P(injection) for each text, batched inference."""
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    tok = AutoTokenizer.from_pretrained(model_dir); model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(dev).eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(texts), batch):
            b = tok(texts[i:i + batch], truncation=True, max_length=max_len, padding=True, return_tensors="pt").to(dev)
            out.append(torch.softmax(model(**b).logits.float(), -1)[:, 1].cpu().numpy())
    return np.concatenate(out) if out else np.array([])


## Stage B. Data: structural splits, evaluation frames, cached released-detector scores, TF-IDF reference

In [ ]:
import importlib, json, random, numpy as np, pandas as pd
import structural_train, data_loaders, interval_panel, rank_metrics
for m in (structural_train, data_loaders, interval_panel, rank_metrics): importlib.reload(m)
from structural_train import load_jsonl, select_config, train_model, score_texts
from data_loaders import load_deepset, load_bipia_categorized, load_notinject, load_jailbreak, BIPIA_HARMFUL, BIPIA_HIJACK, BIPIA_MALICIOUS
from interval_panel import three_thresholds, fnr, fpr, KEYS
from rank_metrics import rank_severity
from sklearn.metrics import roc_auc_score

SPL = {sp: load_jsonl(f'data/structural/{sp}.jsonl') for sp in ('train', 'val', 'test')}
print('structural splits:', {k: len(v) for k, v in SPL.items()})

deepset = load_deepset(); bip = load_bipia_categorized(per_category=30, categories=BIPIA_MALICIOUS); noti = load_notinject(); jb = load_jailbreak()
EVAL = {'bipia': bip.text.tolist(), 'notinject': noti.text.tolist(), 'deepset': deepset.text.tolist(), 'jailbreak': jb.text.tolist(),
        'structural_test': [r['text'] for r in SPL['test']]}
Y = {'bipia': bip.label.values, 'deepset': deepset.label.values, 'jailbreak': jb.label.values, 'structural_test': np.array([r['label'] for r in SPL['test']])}
HIJ = bip.meta.isin(BIPIA_HIJACK).values; HARM = bip.meta.isin(BIPIA_HARMFUL).values; HOST = bip.label.values == 0
TEST_FORM = np.array([r['form'] for r in SPL['test']])

TAGS = ['protectai_v2', 'prompt_guard_2', 'prompt_guard_2_22m']
LABEL = {'protectai_v2': 'ProtectAI-v2', 'prompt_guard_2': 'Prompt-Guard-2 (86M)', 'prompt_guard_2_22m': 'Prompt-Guard-2 (22M)'}
KEYMAP = {'bipia': 'bipia_cat', 'notinject': 'notinject', 'deepset': 'deepset', 'jailbreak': 'jailbreak'}
SCORES = {}
for tag in TAGS:
    SCORES[LABEL[tag]] = {}
    for s, key in KEYMAP.items():
        fp = f'data/score_{tag}_{key}.npy'; assert os.path.exists(fp), f'missing {fp}'
        a = np.load(fp); assert len(a) == len(EVAL[s]), f'{fp}: {len(a)} vs {len(EVAL[s])}'; SCORES[LABEL[tag]][s] = a

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=200000, sublinear_tf=True)
Xtr = vec.fit_transform([r['text'] for r in SPL['train']]); lr = LogisticRegression(C=1.0, max_iter=2000).fit(Xtr, [r['label'] for r in SPL['train']])
SCORES['TF-IDF reference'] = {s: lr.predict_proba(vec.transform(t))[:, 1] for s, t in EVAL.items()}
print('comparators ready:', list(SCORES.keys()))

In [ ]:
%%writefile src/structural_train.py
"""Fine-tune and score a payload-decoupled structural detector on the notebook-17 dataset."""
from __future__ import annotations
import json, os, inspect, random, math
import numpy as np

MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LEN = 512


def load_jsonl(path):
    return [json.loads(l) for l in open(path, encoding="utf-8")]


def select_config(records, config, rng):
    pos = [r for r in records if r["label"] == 1]
    neg = [r for r in records if r["label"] == 0]
    if config == "full":
        return pos + neg
    if config == "ablation_no_twins":
        return pos + [r for r in neg if r["form"] in ("host_only", "declarative_insert")]
    if config == "full_sizematched":
        target = sum(r["form"] in ("host_only", "declarative_insert") for r in neg)
        by_form = {}
        for r in neg: by_form.setdefault(r["form"], []).append(r)
        total = len(neg); out = []
        for form, rs in by_form.items():
            k = int(round(target * len(rs) / total)); rng.shuffle(rs); out += rs[:k]
        return pos + out
    raise ValueError(config)


def class_weights(records):
    n1 = sum(r["label"] for r in records); n0 = len(records) - n1
    return [len(records) / (2 * n0), len(records) / (2 * n1)]


def train_model(train_records, val_records, out_dir, seed=0, epochs=2, lr=2e-5, batch=8, accum=2, max_len=MAX_LEN):
    """Fine-tune with a class-weighted loss; select the epoch with the best val AUROC. Returns the best-model dir."""
    import torch
    from datasets import Dataset
    from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, set_seed)
    from sklearn.metrics import roc_auc_score
    set_seed(seed)
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    def enc(b): return tok(b["text"], truncation=True, max_length=max_len)
    tr = Dataset.from_list([{"text": r["text"], "label": int(r["label"])} for r in train_records]).map(enc, batched=True, remove_columns=["text"])
    va = Dataset.from_list([{"text": r["text"], "label": int(r["label"])} for r in val_records]).map(enc, batched=True, remove_columns=["text"])
    w = torch.tensor(class_weights(train_records), dtype=torch.float)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    model = model.float()   # transformers 5.x can load in the checkpoint dtype; fp16 mixed precision needs fp32 master weights

    class WeightedTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.pop("labels")
            outputs = model(**inputs)
            loss = torch.nn.functional.cross_entropy(outputs.logits.float(), labels, weight=w.to(outputs.logits.device))
            return (loss, outputs) if return_outputs else loss

    def metrics(ep):
        logits, labels = ep.predictions, ep.label_ids
        if isinstance(logits, (tuple, list)): logits = logits[0]
        p = torch.softmax(torch.tensor(np.asarray(logits)).float(), -1)[:, 1].numpy()
        return {"auroc": float(roc_auc_score(labels, p))}

    params = inspect.signature(TrainingArguments.__init__).parameters
    strat_key = "eval_strategy" if "eval_strategy" in params else "evaluation_strategy"
    total_steps = math.ceil(len(tr) / (batch * accum)) * epochs
    warm = {"warmup_ratio": 0.06} if "warmup_ratio" in params else {"warmup_steps": int(0.06 * total_steps)}
    args = TrainingArguments(output_dir=out_dir, per_device_train_batch_size=batch, per_device_eval_batch_size=32,
                             gradient_accumulation_steps=accum, num_train_epochs=epochs, learning_rate=lr, weight_decay=0.01,
                             fp16=torch.cuda.is_available(), save_strategy="epoch", load_best_model_at_end=True,
                             metric_for_best_model="auroc", greater_is_better=True, save_total_limit=1, logging_steps=50,
                             report_to="none", seed=seed, **warm, **{strat_key: "epoch"})
    tok_key = "processing_class" if "processing_class" in inspect.signature(Trainer.__init__).parameters else "tokenizer"
    trainer = WeightedTrainer(model=model, args=args, train_dataset=tr, eval_dataset=va, compute_metrics=metrics, **{tok_key: tok})
    trainer.train()
    trainer.save_model(os.path.join(out_dir, "best")); tok.save_pretrained(os.path.join(out_dir, "best"))
    json.dump({"best_metric": trainer.state.best_metric, "log": trainer.state.log_history}, open(os.path.join(out_dir, "train_log.json"), "w"))
    return os.path.join(out_dir, "best")


def score_texts(model_dir, texts, batch=32, max_len=MAX_LEN):
    """P(injection) for each text, batched inference."""
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    tok = AutoTokenizer.from_pretrained(model_dir); model = AutoModelForSequenceClassification.from_pretrained(model_dir).float().to(dev).eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(texts), batch):
            b = tok(texts[i:i + batch], truncation=True, max_length=max_len, padding=True, return_tensors="pt").to(dev)
            out.append(torch.softmax(model(**b).logits.float(), -1)[:, 1].cpu().numpy())
    return np.concatenate(out) if out else np.array([])

In [ ]:
import importlib, structural_train; importlib.reload(structural_train)
from structural_train import load_jsonl, select_config, train_model, score_texts
print('reloaded; fp32 cast present in loaded module:', 'model.float()' in open('src/structural_train.py').read())

## Stage C. Train the three configurations (skipped for any config whose best model already exists)

In [ ]:
CONFIGS = ['full', 'ablation_no_twins', 'full_sizematched']
MODEL_DIR = {}
rng = random.Random(0)
for cfg in CONFIGS:
    best = f'data/structural/model_{cfg}/best'
    if os.path.exists(os.path.join(best, 'config.json')):
        print(f'{cfg}: trained model found, skipping'); MODEL_DIR[cfg] = best; continue
    recs = select_config(SPL['train'], cfg, rng)
    n1 = sum(r['label'] for r in recs)
    print(f'{cfg}: training on {len(recs)} records ({n1} positive, {len(recs) - n1} negative)')
    MODEL_DIR[cfg] = train_model(recs, SPL['val'], f'data/structural/model_{cfg}', seed=0, epochs=2)
    log = json.load(open(f'data/structural/model_{cfg}/train_log.json')); print(f'   best val AUROC: {log["best_metric"]:.4f}')
print('models:', MODEL_DIR)

## Stage D. Score every evaluation set with each model (cached)

In [ ]:
for cfg in CONFIGS:
    name = f'structural ({cfg})'; SCORES[name] = {}
    for s, texts in EVAL.items():
        fp = f'data/structural/scores/{cfg}_{s}.npy'
        if os.path.exists(fp): SCORES[name][s] = np.load(fp)
        else:
            print('scoring', cfg, s, len(texts)); a = score_texts(MODEL_DIR[cfg], texts); np.save(fp, a); SCORES[name][s] = a
        assert len(SCORES[name][s]) == len(texts)
print('scored:', [k for k in SCORES if k.startswith('structural')])

## Stage E. Evaluation tables

In [ ]:
def pauc(y, p, max_fpr=0.05):
    return float(roc_auc_score(y, p, max_fpr=max_fpr))

rows, irows = [], []
for name, S in SCORES.items():
    pb = S['bipia']; hosts = pb[HOST]; hij = pb[HIJ]; harm = pb[HARM]
    r = dict(model=name,
             AUROC_hijack=roc_auc_score(np.r_[np.ones(len(hij)), np.zeros(len(hosts))], np.r_[hij, hosts]),
             pAUC05_hijack=pauc(np.r_[np.ones(len(hij)), np.zeros(len(hosts))], np.r_[hij, hosts]),
             AUROC_harmful=roc_auc_score(np.r_[np.ones(len(harm)), np.zeros(len(hosts))], np.r_[harm, hosts]),
             AUROC_direct=roc_auc_score(Y['deepset'], S['deepset']),
             AUROC_jailbreak=roc_auc_score(Y['jailbreak'], S['jailbreak']),
             notinject_frac_over_0p5=float((S['notinject'] >= 0.5).mean()))
    rows.append(r)
    for cal_name, cal in (('bipia_hosts', hosts), ('deepset_benigns', S['deepset'][Y['deepset'] == 0])):
        T = three_thresholds(cal)
        ir = dict(model=name, calibration=cal_name, N_cal=int(len(cal)), **{f't_{k}': T[k] for k in KEYS})
        for k in KEYS:
            ir[f'FNR_hijack_{k}'] = fnr(hij, T[k]); ir[f'FNR_harmful_{k}'] = fnr(harm, T[k])
            ir[f'docFPR_{k}'] = fpr(hosts, T[k]); ir[f'notinjectFPR_{k}'] = fpr(S['notinject'], T[k])
        miss = hij < T['guar']
        ir['R_hosts_mean_on_hijack_misses'] = float(rank_severity(hij[miss], hosts).mean()) if miss.sum() else np.nan
        irows.append(ir)
EV = pd.DataFrame(rows); EV.to_csv('reports/structural_detector_eval.csv', index=False)
IV = pd.DataFrame(irows); IV.to_csv('reports/structural_detector_interval.csv', index=False)
pd.set_option('display.width', 250)
print('=== threshold-free: AUROC on the deployment distribution (BIPIA documents), direct and jailbreak for completeness ===')
print(EV.round(3).to_string(index=False))
print('\n=== interval protocol, calibration on the 778 BIPIA hosts (document pool) ===')
print(IV[IV.calibration == 'bipia_hosts'][['model','t_guar','FNR_hijack_lo','FNR_hijack_hi','FNR_hijack_guar','FNR_harmful_guar','docFPR_guar','notinjectFPR_guar','R_hosts_mean_on_hijack_misses']].round(3).to_string(index=False))
print('\n=== interval protocol, calibration on deepset benigns (the preprint protocol) ===')
print(IV[IV.calibration == 'deepset_benigns'][['model','t_lo','t_hi','t_guar','FNR_hijack_lo','FNR_hijack_hi','FNR_hijack_guar','docFPR_lo','docFPR_hi','docFPR_guar']].round(3).to_string(index=False))

# held-out structural test split, per form, for the trained models and the TF-IDF reference
trows = []
for name in [k for k in SCORES if k.startswith('structural')] + ['TF-IDF reference']:
    p = SCORES[name]['structural_test']; y = Y['structural_test']; emb = TEST_FORM == 'embedded'
    r = dict(model=name, AUROC_all=roc_auc_score(y, p))
    for form in ['standalone', 'host_only', 'declarative_insert', 'content_imperative_insert']:
        m = emb | (TEST_FORM == form); r[f'AUROC_emb_vs_{form}'] = roc_auc_score(y[m], p[m])
    trows.append(r)
TS = pd.DataFrame(trows); TS.to_csv('reports/structural_detector_testsplit.csv', index=False)
print('\n=== held-out structural test split (unseen template families, unseen hosts) ===')
print(TS.round(3).to_string(index=False))

## Stage F. Figure: ROC on hijack documents versus hosts

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
fig, ax = plt.subplots(figsize=(6.5, 5.5))
y = np.r_[np.ones(int(HIJ.sum())), np.zeros(int(HOST.sum()))]
for name, S in SCORES.items():
    p = np.r_[S['bipia'][HIJ], S['bipia'][HOST]]; f, t, _ = roc_curve(y, p)
    ax.plot(f, t, lw=2 if name.startswith('structural (full)') else 1.2, label=f'{name} (AUROC {roc_auc_score(y, p):.2f})')
ax.plot([0, 1], [0, 1], 'k--', lw=0.8); ax.axvline(0.01, color='grey', ls=':', lw=0.8)
ax.set_xlabel('false-positive rate on BIPIA host documents'); ax.set_ylabel('true-positive rate on hijack injections')
ax.set_title('Hijack injection versus benign host documents'); ax.legend(fontsize=7, loc='lower right'); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig('figures/structural_detector_roc.png', dpi=150); plt.show()

## Stage G. Log and commit

In [ ]:
from reslog import log_result
summary = ('AUROC table:\n' + EV.round(3).to_string(index=False) + '\n\nInterval (bipia_hosts calibration):\n'
           + IV[IV.calibration=='bipia_hosts'][['model','FNR_hijack_guar','FNR_harmful_guar','docFPR_guar','notinjectFPR_guar']].round(3).to_string(index=False)
           + '\n\nStructural test split:\n' + TS.round(3).to_string(index=False))
log_result('nb18: structural detector fine-tune, ablation, deployment-distribution evaluation', summary, csv_df=EV, csv_name='structural_detector_eval.csv')
print(summary)

# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
r = subprocess.run(["python", "tools/commit_cell.py",
                    "nb18: structural detector (full, ablation, size-matched), evaluation on BIPIA/NotInject/direct/jailbreak, ROC figure; add src/structural_train.py"],
                   capture_output=True, text=True)
print(r.stdout); print(r.stderr)